In [4]:
import requests
from bs4 import BeautifulSoup
import time 
import sqlite3

In [6]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import re

# 1. データベースのセットアップ (SQLite)
dbname = 'pitcher_injuries.db'
conn = sqlite3.connect(dbname)
cur = conn.cursor()

# テーブル作成 (もし存在しなければ)
# id, 名前, 日付, 怪我の種類
cur.execute('''
    CREATE TABLE IF NOT EXISTS injuries (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        date TEXT,
        injury_type TEXT
    )
''')
conn.commit()

# 2. スクレイピング設定
url = 'https://www.niwaka-yakyu.info/2025-l-npb-il-list/' 

# 実際のサイトへのアクセス (ユーザーエージェントを設定しておくとブロックされにくいです)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status() # エラーチェック
    soup = BeautifulSoup(response.content, 'html.parser')

    # 3. データ抽出処理
    # HTML構造: figure.wp-block-table -> table -> tbody -> tr の順
    table = soup.select_one('figure.wp-block-table table tbody')
    
    if not table:
        print("テーブルが見つかりませんでした。")
        exit()

    rows = table.find_all('tr')

    current_pitcher_name = None # 選手名を一時保存する変数
    data_to_save = []

    for row in rows:
        # --- パターンA: 選手名の行かチェック ---
        # 特徴: badge-blueクラスがある、または背番号のような数字がある
        badge = row.find('span', class_='badge-blue')
        name_tag = row.find('strong')

        if badge and name_tag:
            # 選手名を更新 (例: "西舘 昂汰")
            current_pitcher_name = name_tag.get_text(strip=True)
            continue # 次の行へ（怪我情報を探しに行く）

        # --- パターンB: 怪我情報の行かチェック ---
        # 特徴: 日付が入っている span class="red" がある
        date_span = row.find('span', class_='red')
        
        if date_span and current_pitcher_name:
            # 日付の抽出 (例: "【24/09/21】" -> "24/09/21")
            raw_date = date_span.get_text(strip=True)
            date = raw_date.replace('【', '').replace('】', '')

            # 怪我の種類の抽出
            # 日付の横にある strong タグが怪我名
            injury_tag = row.find('strong')
            injury_name = injury_tag.get_text(strip=True) if injury_tag else "不明"

            # データをリストに追加
            data_to_save.append((current_pitcher_name, date, injury_name))
            
            print(f"取得: {current_pitcher_name} - {date} - {injury_name}")

    # 4. データベースへの保存
    if data_to_save:
        cur.executemany('INSERT INTO injuries (name, date, injury_type) VALUES (?, ?, ?)', data_to_save)
        conn.commit()
        print(f"合計 {len(data_to_save)} 件のデータを保存しました。")
    else:
        print("保存するデータが見つかりませんでした。")

except Exception as e:
    print(f"エラーが発生しました: {e}")

finally:
    conn.close()


取得: 西舘　昂汰 - 24/09/21 - 右肘じん帯再建手術（トミー・ジョン手術）
合計 1 件のデータを保存しました。


In [10]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import re

# 1. データベース設定
dbname = 'pitcher_injuries.db'
conn = sqlite3.connect(dbname)
cur = conn.cursor()

cur.execute('''
    CREATE TABLE IF NOT EXISTS injuries (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        date TEXT,
        injury_type TEXT
    )
''')
conn.commit()

# 2. スクレイピング設定
url = 'https://www.niwaka-yakyu.info/2025-l-npb-il-list/' 
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')

    # 【変更点1】ページ内の「全ての」テーブルブロックを取得する
    tables = soup.find_all('figure', class_='wp-block-table')
    
    if not tables:
        print("テーブルが見つかりませんでした。")
        exit()

    data_to_save = []
    
    # 全てのテーブルをループ処理
    for table in tables:
        # そのテーブル内の行(tr)を取得
        rows = table.find_all('tr')
        current_pitcher_name = None 

        for row in rows:
            # --- 名前行の判定 ---
            # badge-blueがある、または strongタグがある行を名前と仮定
            badge = row.find('span', class_='badge-blue')
            name_tag = row.find('strong')

            if badge and name_tag:
                current_pitcher_name = name_tag.get_text(strip=True)
                continue 

            # --- 怪我情報行の判定 ---
            # 名前が取得済みで、かつ strongタグ（怪我名）がある場合
            if current_pitcher_name:
                # 行全体のテキストを取得
                row_text = row.get_text()
                
                # 【変更点2】正規表現で日付 【MM/DD】 または 【YY/MM/DD】 を探す
                # 画像を見ると 【10/20】 や 【24/09/21】 の形式がある
                date_match = re.search(r'【(.*?)】', row_text)
                
                # 怪我名の取得 (最初のstrongタグの中身を採用)
                injury_tag = row.find('strong')
                
                # 怪我名か日付のどちらかが見つかればデータとして扱う
                if injury_tag or date_match:
                    date = date_match.group(1) if date_match else "日付不明"
                    injury_name = injury_tag.get_text(strip=True) if injury_tag else "詳細不明"

                    # データをリストに追加
                    data_to_save.append((current_pitcher_name, date, injury_name))
                    print(f"取得: {current_pitcher_name} - {date} - {injury_name}")

    # 3. 保存処理
    if data_to_save:
        # 重複を防ぎたい場合は一度テーブルを空にするか、ロジックを追加してください
        cur.executemany('INSERT INTO injuries (name, date, injury_type) VALUES (?, ?, ?)', data_to_save)
        conn.commit()
        print(f"\n完了: 合計 {len(data_to_save)} 件のデータを保存しました。")
    else:
        print("データが見つかりませんでした。")

except Exception as e:
    print(f"エラー: {e}")

finally:
    conn.close()

取得: 西舘　昂汰 - 24/09/21 - 右肘じん帯再建手術（トミー・ジョン手術）
取得: 茂木　栄五郎 - 07/17 - 左膝半月板手術
取得: 塩見　泰隆 - 04/18 - 左膝前十字じん帯の手術
取得: ドミンゴ・サンタナ - 06/20 - 右前腕部打撲
取得: 下村　海翔 - 24/04/11 - 右肘内側側副じん帯再建術（トミージョン手術）
取得: 榮枝　裕貴 - 10/24 - 右尺骨骨折観血的手術
取得: 百崎　蒼生 - 08/07 - 下顎骨骨折における整復固定術
取得: フォスター・グリフィン - 08/03 - 膝の違和感で登録抹消（違和感は7月から）
取得: 赤星　優志 - 09/14 - 右肩痛
取得: 京本　眞 - 08/18 - 右肘内側側副靭帯再建術（TJ手術）
取得: 吉川　尚輝 - 10/27 - 両側関節鏡視下股関節唇形成術
取得: 甲斐　拓也 - 08/23 - 右中指中手骨頭骨折
取得: 喜多　隆介 - 08/15 - 右膝外側半月板縫合術及び制動術
取得: 大瀬良　大地 - 10/03 - 右肘関節授動術、関節形成術、滑膜切除術
取得: 森下　暢仁 - 08/24 - 右肩炎症
取得: 黒原　拓未 - 02/06 - 左膝違和感
取得: 中村　奨成 - 10/07 - 右足首の手術（遊離体摘出術）
取得: 梅津　晃大 - 2024/12月下旬 - チームドクターから「肩に大きな問題がある」
取得: 森　博人 - 03/03 - 尺側側副靭帯再建術（トミージョン手術）
取得: 梅津　晃大 - 2024/12月下旬 - チームドクターから「肩に大きな問題がある」
取得: 森　博人 - 03/03 - 尺側側副靭帯再建術（トミージョン手術）
取得: 石田　健大 - 24/06/06 - 左肩の肉離れ
取得: 大貫　晋一 - 07/11 - 上半身違和感
取得: 小園　健太 - 08/10以降 - コンディション不良
取得: 松本　隆之介 - 04/24 - 右前十字靱帯再建術の手術
取得: 坂口　翔颯 - 24/12/12 - 右肘の靱帯再建術(TJ手術)
取得: 浜地　真澄 - 07/15 - 右肘肘頭固定術、右肘クリーニングの手術
取得: ローワン・ウィック - 09/26 - 上半身のコンディション不良
取得: タ

In [17]:
# stats_pit.csvの中身を5行確認
import pandas as pd

pd.set_option('display.max_columns', None)   # 全ての列を表示
pd.set_option('display.max_rows', None)      # 全ての行を表示（必要なら）
pd.set_option('display.max_colwidth', None)

df = pd.read_csv('play_info.csv',  encoding="cp932")
df.head()

/var/folders/56/v1rqc2cn6rs7t55b3ql05_0r0000gn/T/ipykernel_10933/3067985060.py:8: DtypeWarning: Columns (125,141) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('play_info.csv',  encoding="cp932")


,game_id,seqno_9,year_id,game_kind_id,game_kind_name,game_date,game_year,game_month,stadium_id,stadium_name,home_team_league_id,home_team_league_name,home_team_id,home_team_name,away_team_league_id,away_team_league_name,away_team_id,away_team_name,play_date,play_time,inning,top_bottom_id,top_bottom_name,pa_of_inning,np_of_pa,pitcher_team_league_id,pitcher_team_league_name,pitcher_team_id,pitcher_team_name,pitcher_id,pitcher_name,pitcher_handedness,is_starter,bf_of_game,np_of_game,batter_team_league_id,batter_team_league_name,batter_team_id,batter_team_name,batter_id,batter_name,batter_handedness,batter_pos,batting_order,is_pinch_hitter,pitch_result,pa_result,run_scored,pickoff_attempt_to,pitch_type_id,pitch_type_name,pitch_type_name_detail,pitch_type_group,pitch_speed,pitch_location_x,pitch_location_y,pitch_target_location_x,pitch_target_location_y,pitch_zone,pitch_zone_side,pitch_zone_height,batted_ball_location_x,batted_ball_location_y,batted_ball_grounded_location_x,batted_ball_grounded_location_y,batted_ball_direction,batted_ball_direction_3,batted_ball_direction_5,batted_ball_distance,batted_ball_pos,batted_ball_sec,batted_ball_type,batted_ball_quality_sec,batted_ball_quality,pre_home_team_score,pre_away_team_score,pre_home_team_score_diff,pre_away_team_score_diff,pre_pitcher_team_score,pre_batter_team_score,pre_pitcher_team_score_diff,pre_batter_team_score_diff,pre_runner_situation,pre_runner_count,pre_ball,pre_strike,pre_out,post_home_team_score,post_away_team_score,post_home_team_score_diff,post_away_team_score_diff,post_pitcher_team_score,post_batter_team_score,post_pitcher_team_score_diff,post_batter_team_score_diff,post_runner_situation,post_runner_count,post_ball,post_strike,post_out,is_risp,count_situation,catcher_id,catcher_name,runner_1b_id,runner_1b_name,runner_2b_id,runner_2b_name,runner_3b_id,runner_3b_name,batter_runner_advanced_to,is_batter_runner_out,runner_1b_advanced_to,is_runner_1b_out,is_runner_1b_sb,is_runner_1b_cs,runner_2b_advanced_to,is_runner_2b_out,is_runner_2b_sb,is_runner_2b_cs,runner_3b_advanced_to,is_runner_3b_out,is_runner_3b_sb,is_runner_3b_cs,fielding_char,fielder_alignment,fielder_1b_id,fielder_1b_name,fielder_2b_id,fielder_2b_name,fielder_3b_id,fielder_3b_name,fielder_ss_id,fielder_ss_name,fielder_lf_id,fielder_lf_name,fielder_cf_id,fielder_cf_name,fielder_rf_id,fielder_rf_name,fielder_dh_id,fielder_dh_name,umpire_id,umpire_name,rec_pitcher_id,rec_pitcher_name,rec_pitcher_handedness,rec_is_starter,rec_batter_id,rec_batter_name,rec_batter_handedness,rec_batter_pos,rec_is_pinch_hitter,is_pitch,is_ball,is_strike,is_called_strike,is_swinging_strike,is_foul,is_swing,is_contact,in_strike_zone,is_opposite_pitch,is_pa,is_ab,is_h,is_1b,is_2b,is_3b,is_hr,tb,rbi,is_bb,is_ibb,is_ubb,is_so,is_hbp,is_sh,is_sf,is_gidp,is_roe,is_fc,is_dp,is_out,is_bbe,is_gb,is_fb,is_ld,is_pu,is_ifh,is_bunt,is_pull,is_straight,is_oppo,is_left,is_right,is_soft,is_med,is_hard,is_productive_out,is_sac_bunt_attempt,is_wp,is_bk,is_pb,is_pk,in_dirt,off_wall,change_situation
0,2021029038,11010101,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,1,セ・リーグ,1,巨人,1,セ・リーグ,2,ヤクルト,2025-03-28,18:19:00,1,1,表,1,1,1,セ・リーグ,1,巨人,1800028,戸郷 翔征,右,1,1,1,1,セ・リーグ,2,ヤクルト,1000100,西川 遥輝,左,9,1,0,ファウル,NaN,0,0,1.0,ストレート,ストレート,直球系,146.0,21.0,177.0,36.0,119.0,2.0,外角,真ん中,-198.0,117.0,NaN,NaN,-59.420773,1.0,1.0,76.661594,NaN,2.0,フライ,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,並行,1000176,甲斐 拓也,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,1400101,岡本 和真,1600059,吉川 尚輝,700003,坂本 勇人,2107874,門脇 誠,2000070,若林 楽人,1961425,ヘルナンデス,2118420,キャベッジ,NaN,NaN,9,敷田 直人,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,1,0,0,1,1,1,1,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2021029038,11010201,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,1,セ・リーグ,1,巨人,1,セ・リーグ,2,ヤクルト,2025-03-28,18:19:22.188,1,1,表,1,2,1,セ・リーグ,1,巨人,1800028,戸郷 翔征,右,1,1,2,1,セ・リーグ,2,ヤクルト,1000100,西川 遥輝,左,9,1,0,凡打,一ゴロ,0,0,4.0,スライダー,スライダー,曲がる系,128.0,-14.0,1

## injuriesテーブル
・誰が、いつ、どこを怪我したか
### injuriesテーブルの中身
injury_id INTEGER PRIMARY KEY AUTOINCREMENT
player_name TEXT, 結合キー(CSVの名前と一致)
injury_date TEXT,

body_part TEXT, 部位：'Elbow'(肘), 'Shoulder'(肩), 'Oblique'(脇腹), 'Leg'(脚)など
side TEXT,      患側：'Right'(右), 'Left'(左)
injury_type TEXT, 種類：'Ligament'(靭帯), 'Muscle'(筋肉/肉離れ), 'Impact'(打撲/骨折)

複合ユニーク制約（同じ日の同じ怪我を重複登録しないため）
UNIQUE(player_name, injury_date, original_text)


## daily_statsテーブル
・play_info.csvから作成
・投手の「その日の負荷」
・詳細なデータから「1試合ごとの集計データ」に加工。
### daily_statsテーブルの中身
pitcher_name
pitcher_handedness
game_date

np_of_game(または行数をカウント):総投球数
inning:イニング数
rest_days:(dateから計算)中何日で登板したか

pitch_speed(の平均と最大)
pitch_tipe_name(これから変化球の割合を計算):


In [18]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import re

# ---------------------------------------------------------
# 1. 分析用タグ付けロジック (部位・左右・種類の自動判定)
# ---------------------------------------------------------
def categorize_injury(text):
    part = "Other"
    side = "Unknown"
    injury_type = "Other"
    
    clean_text = text.replace("じん帯", "靭帯")

    # --- 患側 (Side) ---
    if "右" in clean_text:
        side = "Right"
    elif "左" in clean_text:
        side = "Left"

    # --- 部位 (Body Part) ---
    if "肘" in clean_text or "トミー" in clean_text or "TJ" in clean_text:
        part = "Elbow"
    elif "肩" in clean_text:
        part = "Shoulder"
    elif any(w in clean_text for w in ["股関節", "足", "膝", "脚", "ハムストリング", "内転筋", "アキレス"]):
        part = "LowerBody"
    elif any(w in clean_text for w in ["脇腹", "腹斜筋", "腰", "背中"]):
        part = "Core"
    elif "指" in clean_text or "手" in clean_text:
        part = "Hand/Finger"

    # --- 種類 (Type) ---
    if any(w in clean_text for w in ["靭帯", "靱帯", "TJ", "トミー"]):
        injury_type = "Ligament"
    elif any(w in clean_text for w in ["骨折", "骨"]):
        injury_type = "Bone"
    elif any(w in clean_text for w in ["クリーニング", "遊離体", "骨棘"]):
        injury_type = "Cleaning"
    elif any(w in clean_text for w in ["肉離れ", "筋損傷", "炎症", "張り", "コンディション不良", "違和感"]):
        injury_type = "Muscle/Fatigue"

    return part, side, injury_type

# ---------------------------------------------------------
# 2. データベース設定
# ---------------------------------------------------------
dbname = 'baseball_analysis.db'
conn = sqlite3.connect(dbname)
cur = conn.cursor()

cur.execute('''
    CREATE TABLE IF NOT EXISTS injuries (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        team TEXT,
        player_name TEXT,
        injury_date TEXT,
        injury_name TEXT,
        body_part TEXT,
        side TEXT,
        injury_type TEXT,
        position TEXT,
        UNIQUE(player_name, injury_date, injury_name)
    )
''')
conn.commit()

# ---------------------------------------------------------
# 3. URLからデータ取得 & 解析
# ---------------------------------------------------------

# ★ここにスクレイピングしたいページのURLを入れてください
target_url = "https://www.niwaka-yakyu.info/2025-l-npb-il-list/" 
# ※もし別のページならここを書き換えてください

# ブラウザのふりをする設定（ブロック回避用）
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
}

try:
    print(f"サイトにアクセス中...: {target_url}")
    response = requests.get(target_url, headers=headers)
    response.raise_for_status() # エラーがあれば停止
    response.encoding = response.apparent_encoding # 日本語の文字化け防止
    
    html_content = response.text
    soup = BeautifulSoup(html_content, 'html.parser')

    # テーブル要素をすべて取得
    tables = soup.find_all('figure', class_='wp-block-table')
    
    if not tables:
        print("テーブルが見つかりませんでした。URLやクラス名を確認してください。")
        exit()

    data_list = []
    print(f"{len(tables)} 個のテーブルが見つかりました。解析を開始します。")

    for table in tables:
        # チーム名とポジション（投手/野手）の判定
        prev_h4 = table.find_previous('h4')
        position_type = prev_h4.get_text(strip=True) if prev_h4 else "不明"
        
        prev_h2 = table.find_previous('h2')
        team_name = prev_h2.get_text(strip=True) if prev_h2 else "チーム不明"

        # 「投手」のテーブルだけ処理する
        if "投手" not in position_type:
            continue

        rows = table.find_all('tr')
        current_player = None

        for row in rows:
            # 名前行チェック
            badge = row.find('span', class_='badge-blue')
            name_tag = row.find('strong')

            if badge and name_tag:
                raw_name = name_tag.get_text(strip=True)
                # 分析用に名前の空白を削除 (例: "西舘　昂汰" -> "西舘昂汰")
                current_player = raw_name.replace('\u3000', '').replace(' ', '')
                continue

            # 怪我情報行チェック
            if current_player:
                text_content = row.get_text()
                
                # 日付抽出: 【MM/DD】または【YY/MM/DD】
                date_match = re.search(r'【(.*?)】', text_content)
                injury_tag = row.find('strong')
                
                if date_match and injury_tag:
                    raw_date = date_match.group(1)
                    injury_name = injury_tag.get_text(strip=True)
                    
                    # 日付の整形 (2024年または2025年を補完)
                    # ここでは簡易的に、年がなければ2024としていますが、
                    # ページの文脈に合わせて適宜2025にするなど調整可能です
                    if raw_date.count('/') == 2:
                        parts = raw_date.split('/')
                        formatted_date = f"20{parts[0]}-{parts[1]}-{parts[2]}" # YY/MM/DD -> 20YY-MM-DD
                    else:
                        # 年がない場合 (MM/DD) -> 今の時期なら2024年の怪我が多いと想定
                        formatted_date = f"2024-{raw_date.replace('/', '-')}"

                    # 自動タグ付け
                    part, side, i_type = categorize_injury(injury_name)

                    data_list.append((
                        team_name, current_player, formatted_date, 
                        injury_name, part, side, i_type, "Pitcher"
                    ))
                    
                    print(f"  OK: {team_name} {current_player} ({formatted_date}) - {part}/{side}")

    # 保存処理
    if data_list:
        cur.executemany('''
            INSERT OR IGNORE INTO injuries 
            (team, player_name, injury_date, injury_name, body_part, side, injury_type, position) 
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', data_list)
        conn.commit()
        print(f"\n完了: {len(data_list)} 件のデータを保存しました。")
    else:
        print("データが見つかりませんでした。")

except Exception as e:
    print(f"エラーが発生しました: {e}")

finally:
    conn.close()

サイトにアクセス中...: https://www.niwaka-yakyu.info/2025-l-npb-il-list/
23 個のテーブルが見つかりました。解析を開始します。
  OK: 東京ヤクルトスワローズ 西舘昂汰 (2024-09-21) - Elbow/Right
  OK: 阪神タイガース 下村海翔 (2024-04-11) - Elbow/Right
  OK: 読売ジャイアンツ フォスター・グリフィン (2024-08-03) - LowerBody/Unknown
  OK: 読売ジャイアンツ 赤星優志 (2024-09-14) - Shoulder/Right
  OK: 読売ジャイアンツ 京本眞 (2024-08-18) - Elbow/Right
  OK: 広島東洋カープ 大瀬良大地 (2024-10-03) - Elbow/Right
  OK: 広島東洋カープ 森下暢仁 (2024-08-24) - Shoulder/Right
  OK: 広島東洋カープ 黒原拓未 (2024-02-06) - LowerBody/Left
  OK: 中日ドラゴンズ 梅津晃大 (2024-2024-12月下旬) - Shoulder/Unknown
  OK: 中日ドラゴンズ 森博人 (2024-03-03) - Elbow/Unknown
  OK: 横浜DeNAベイスターズ 石田健大 (2024-06-06) - Shoulder/Left
  OK: 横浜DeNAベイスターズ 大貫晋一 (2024-07-11) - Other/Unknown
  OK: 横浜DeNAベイスターズ 小園健太 (2024-08-10以降) - Other/Unknown
  OK: 横浜DeNAベイスターズ 松本隆之介 (2024-04-24) - Hand/Finger/Right
  OK: 横浜DeNAベイスターズ 坂口翔颯 (2024-12-12) - Elbow/Right
  OK: 横浜DeNAベイスターズ 浜地真澄 (2024-07-15) - Elbow/Right
  OK: 横浜DeNAベイスターズ ローワン・ウィック (2024-09-26) - Other/Unknown
  OK: オリックス・バファローズ アンダーソン・エスピ


完了: 42 件のデータを保存しました。
